# Ray tasks

In [3]:
import os
os.environ["RAY_ENABLE_UV_RUN_RUNTIME_ENV"] = "0"

In [4]:
import ray
import time
import numpy as np
import socket
import sys

In [5]:
ray.init(address="auto")

2026-06-03 14:08:03,339	INFO worker.py:1814 -- Connecting to existing Ray cluster at address: 172.31.34.33:6379...
2026-06-03 14:08:03,371	INFO worker.py:2003 -- Connected to Ray cluster. View the dashboard at http://172.31.34.33:8265 
/home/ubuntu/ray_demo/.venv/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Python version:,3.12.3
Ray version:,2.55.1
Dashboard:,http://172.31.34.33:8265


In [6]:
database = ["learning", "Ray", "for", "distributed", "data", "processing"]

In [7]:
def retrieve(idx):
    time.sleep(idx / 10.)
    return idx, database[idx]

In [10]:
%%time

t0 = time.time()
results = [retrieve(idx) for idx in range(6)]
t1 = time.time()
print(results)
print(t1-t0, "seconds")

[(0, 'learning'), (1, 'Ray'), (2, 'for'), (3, 'distributed'), (4, 'data'), (5, 'processing')]
1.5013227462768555 seconds
CPU times: user 1.54 ms, sys: 6.21 ms, total: 7.76 ms
Wall time: 1.5 s


In [13]:
@ray.remote
def retrieve_task(idx):
    return retrieve(idx)

In [15]:
t0 = time.time()
results = [retrieve_task.remote(idx) for idx in range(6)]
t1 = time.time()
#print(results)
print(t1-t0, "seconds")

0.0038847923278808594 seconds


In [17]:
%%timeit

t0 = time.time()
res_refs = [retrieve_task.remote(idx) for idx in range(6)]
results = ray.get(res_refs) # serves as a barrier
t1 = time.time()

print(results)
print(t1-t0, "seconds")

[(0, 'learning'), (1, 'Ray'), (2, 'for'), (3, 'distributed'), (4, 'data'), (5, 'processing')]
0.6162335872650146 seconds
[(0, 'learning'), (1, 'Ray'), (2, 'for'), (3, 'distributed'), (4, 'data'), (5, 'processing')]
0.5641462802886963 seconds
[(0, 'learning'), (1, 'Ray'), (2, 'for'), (3, 'distributed'), (4, 'data'), (5, 'processing')]
0.5588157176971436 seconds
[(0, 'learning'), (1, 'Ray'), (2, 'for'), (3, 'distributed'), (4, 'data'), (5, 'processing')]
0.5128393173217773 seconds
[(0, 'learning'), (1, 'Ray'), (2, 'for'), (3, 'distributed'), (4, 'data'), (5, 'processing')]
0.5614843368530273 seconds
[(0, 'learning'), (1, 'Ray'), (2, 'for'), (3, 'distributed'), (4, 'data'), (5, 'processing')]
0.5148193836212158 seconds
[(0, 'learning'), (1, 'Ray'), (2, 'for'), (3, 'distributed'), (4, 'data'), (5, 'processing')]
0.6326556205749512 seconds
[(0, 'learning'), (1, 'Ray'), (2, 'for'), (3, 'distributed'), (4, 'data'), (5, 'processing')]
0.5112707614898682 seconds
551 ms ± 40.3 ms per loop (mean 

In [18]:
db_refs = ray.put(database)

In [20]:
db_refs

ObjectRef(00ffffffffffffffffffffffffffffffffffffff0700000001e1f505)

In [21]:
@ray.remote
def retrieve_task_by_ref(idx, db_refs):
    time.sleep(idx / 10.)
    return idx, db_refs[idx], socket.gethostbyname(socket.gethostname())

In [22]:
t0 = time.time()
res_refs = [retrieve_task_by_ref.remote(idx, db_refs) for idx in range(6)]
results = ray.get(res_refs) # serves as a barrier
t1 = time.time()

print(results)
print(t1-t0, "seconds")

[(0, 'learning', '172.31.34.33'), (1, 'Ray', '172.31.34.33'), (2, 'for', '172.31.33.70'), (3, 'distributed', '172.31.33.69'), (4, 'data', '172.31.34.33'), (5, 'processing', '172.31.33.70')]
0.7440190315246582 seconds


In [23]:
data = np.random.randint(10, size=[10_000])
data

array([6, 0, 3, ..., 5, 1, 8], shape=(10000,))

In [57]:
num_partitions = 12
partition_sz = len(data) // num_partitions
print(partition_sz)
input_buckets = [data[i * partition_sz : (i+1) * partition_sz] for i in range(num_partitions)]

833


In [58]:
@ray.remote
def upstream_task(input):
    return input, socket.gethostbyname(socket.gethostname())

In [59]:
@ray.remote
def downstream_task(input):
    intermediate_res, hostname = input
    return np.sum(intermediate_res), hostname, socket.gethostbyname(socket.gethostname())

In [60]:
input_buckets

[array([6, 0, 3, 2, 8, 9, 9, 6, 0, 3, 1, 5, 1, 5, 5, 8, 9, 5, 4, 1, 3, 6,
        3, 7, 5, 8, 9, 7, 8, 1, 5, 8, 4, 1, 8, 8, 6, 9, 2, 2, 1, 7, 3, 2,
        7, 4, 5, 0, 7, 8, 9, 0, 7, 6, 5, 7, 6, 4, 3, 8, 4, 4, 1, 6, 0, 4,
        0, 8, 0, 4, 8, 2, 9, 5, 6, 6, 4, 6, 9, 0, 1, 9, 8, 2, 0, 4, 8, 3,
        9, 5, 9, 1, 1, 0, 0, 3, 7, 8, 5, 2, 4, 3, 4, 5, 4, 4, 9, 7, 1, 1,
        9, 7, 4, 6, 1, 3, 0, 2, 9, 7, 4, 9, 7, 7, 0, 5, 1, 6, 0, 4, 4, 4,
        8, 4, 3, 7, 7, 5, 7, 8, 8, 4, 1, 5, 6, 8, 5, 2, 1, 9, 6, 5, 7, 7,
        2, 0, 1, 3, 9, 6, 0, 6, 7, 9, 5, 8, 7, 8, 1, 0, 1, 0, 7, 2, 3, 9,
        8, 8, 0, 4, 8, 4, 9, 4, 1, 1, 3, 8, 7, 9, 6, 7, 3, 1, 8, 5, 7, 3,
        1, 5, 4, 1, 3, 1, 0, 4, 9, 6, 0, 5, 6, 5, 3, 3, 1, 3, 8, 3, 2, 9,
        0, 9, 3, 3, 8, 0, 3, 5, 9, 7, 0, 4, 1, 8, 6, 3, 7, 8, 2, 5, 3, 6,
        5, 6, 6, 3, 6, 4, 5, 7, 8, 5, 4, 0, 5, 0, 8, 5, 9, 0, 1, 4, 8, 1,
        6, 1, 8, 0, 4, 3, 7, 4, 4, 8, 0, 8, 7, 4, 3, 0, 0, 9, 5, 6, 4, 0,
        6, 8, 8, 2, 0, 5, 7, 4, 4, 3, 

In [61]:
obj_refs = [upstream_task.remote(input) for input in input_buckets]
final_refs = [downstream_task.remote(obj_ref) for obj_ref in obj_refs]

print(ray.get(final_refs))

[(np.int64(3882), '172.31.34.33', '172.31.33.70'), (np.int64(3778), '172.31.33.70', '172.31.33.70'), (np.int64(3708), '172.31.33.69', '172.31.33.70'), (np.int64(3650), '172.31.33.70', '172.31.33.70'), (np.int64(3908), '172.31.33.70', '172.31.33.70'), (np.int64(3793), '172.31.33.69', '172.31.33.70'), (np.int64(3748), '172.31.33.70', '172.31.33.70'), (np.int64(3756), '172.31.33.70', '172.31.34.33'), (np.int64(3761), '172.31.33.69', '172.31.33.70'), (np.int64(3569), '172.31.34.33', '172.31.33.70'), (np.int64(3678), '172.31.33.70', '172.31.34.33'), (np.int64(3674), '172.31.33.69', '172.31.33.70')]


# Ray actors

In [76]:
@ray.remote
class Actor:
    def __init__(self):
        self.counts = 0
    def increment(self):
        self.counts += 1
    def counts(self):
        return self.counts
    def where_am_i(self):
        return socket.gethostbyname(socket.gethostname())

In [77]:
@ray.remote
def downstream_task_actor(input, actor):
    intermediate_result, hostname = input
    #ray.get(actor.increment.remote())  # wait for actor update
    actor.increment.remote()          # fire and forget
    return np.sum(intermediate_result), hostname

In [78]:
actor = Actor.remote()

In [79]:
t0 = time.time()
upstream_task_result_refs = [upstream_task.remote(input) for input in input_buckets]
downstream_task_result_refs = [
    downstream_task_actor.remote(upstream_task_result_ref, actor) for upstream_task_result_ref in upstream_task_result_refs
]
final_results = ray.get(downstream_task_result_refs)
t1 = time.time()

t2 = time.time()
print(ray.get(actor.counts.remote()))
t3 = time.time()

#print(upstream_task_result_refs)
#print(downstream_task_result_refs)
print(final_results)

print(t1-t0, "seconds")
print(t3-t2, "seconds")

12
[(np.int64(3882), '172.31.34.33'), (np.int64(3778), '172.31.34.33'), (np.int64(3708), '172.31.33.70'), (np.int64(3650), '172.31.33.69'), (np.int64(3908), '172.31.34.33'), (np.int64(3793), '172.31.33.69'), (np.int64(3748), '172.31.33.69'), (np.int64(3756), '172.31.33.70'), (np.int64(3761), '172.31.33.69'), (np.int64(3569), '172.31.33.69'), (np.int64(3678), '172.31.33.69'), (np.int64(3674), '172.31.33.70')]
0.28536319732666016 seconds
2.6998090744018555 seconds
